# Eglantine → PS5
Compila **Eglantine** (emulador de Switch 2, siempre su último commit de la rama `nx2`) con el frontend de **ProsperoEden** encima, con nombre y colores de Eglantine (rosa), para PS5 con jailbreak.

Ejecuta las celdas en orden. Todo se guarda en Drive en `eglantine-ps5/` (no toca la carpeta del notebook anterior). Si Colab se reinicia, repite la 1 y la 2: el estado guardado hace que la compilación sea incremental.

Resultado: carpeta `PPSA99308` (copiar a `/data/homebrew/`). Datos en la PS5: `/data/prosperoeden/` (keys, firmware, roms), **compartidos** con ProsperoEden: puedes tener ambos instalados.

In [ ]:
# 1 · Preparar: Drive, repo, estado guardado y herramientas del sistema
import os
from google.colab import drive
drive.mount('/content/drive')
DRV = '/content/drive/MyDrive/eglantine-ps5'; os.makedirs(DRV, exist_ok=True)
REPO = '/content/repo-eglantine'
if os.path.isdir(REPO + '/.git'):
    !git -C {REPO} fetch -q origin eglantine
    !git -C {REPO} checkout -q -B eglantine FETCH_HEAD
else:
    !git clone -q --single-branch --branch eglantine https://github.com/viejunerd/eden-nx2-ps5.git {REPO}
!git -C {REPO} log -1 --oneline
print('En Drive:')
!ls -lh "{DRV}"
state = DRV + '/estado-eglantine.zip'
if os.path.isdir('/content/work-eglantine'):
    print('Ya hay /content/work-eglantine en este entorno; no restauro nada.')
elif os.path.exists(state):
    print('Restaurando el estado guardado (varios GB, unos minutos)…')
    !apt-get -qq install -y unzip >/dev/null
    !unzip -q -o "{state}" -d /
    !ldconfig
else:
    print('Sin estado previo: la primera compilación es completa (~3 h).')
!bash {REPO}/scripts/setup-host.sh

In [ ]:
# 2 · Compilar en segundo plano (siempre parte del último commit de Eglantine)
import os, subprocess, glob, shutil
def alive(pid):
    out = subprocess.run(['ps','-o','stat=','-p',str(pid)],capture_output=True,text=True).stdout.strip()
    return bool(out) and not out.startswith('Z')
pid = int(open('/content/eglantine.pid').read()) if os.path.exists('/content/eglantine.pid') else 0
if pid and alive(pid) and not os.path.exists('/content/eglantine.exit'):
    print('Ya hay un build en marcha (PID %d). Mira el progreso en la celda 3.' % pid)
else:
    if os.path.exists('/content/eglantine.exit'): os.remove('/content/eglantine.exit')
    # una configuración de Mesa a medias (sin build.ninja) se borra para repetirla
    for d in glob.glob('/content/work-eglantine/mihawk-vulkan-review/.deps/work/radv-*build*'):
        if os.path.isdir(d) and not os.path.exists(d + '/build.ninja'): shutil.rmtree(d, ignore_errors=True)
    env = dict(os.environ, WORKDIR='/content/work-eglantine', SKIP_APT='1', JOBS=str(os.cpu_count()),
               LLVM_CONFIG='/usr/local/bin/llvm-config', CCACHE_DIR='/root/.cache/ccache', CCACHE_MAXSIZE='20G')
    p = subprocess.Popen(['bash','-c','/content/repo-eglantine/scripts/build-eglantine.sh > /content/eglantine.log 2>&1; echo $? > /content/eglantine.exit'],
                         env=env, start_new_session=True, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    open('/content/eglantine.pid','w').write(str(p.pid))
    print('Build lanzado (PID %d). Ejecuta la celda 3 para ver el progreso.' % p.pid)

In [ ]:
# 3 · Progreso (o parar el build: pon PARAR = True y ejecuta)
PARAR = False  # @param {type:"boolean"}
import os, re, signal, subprocess, time
def alive(pid):
    out = subprocess.run(['ps','-o','stat=','-p',str(pid)],capture_output=True,text=True).stdout.strip()
    return bool(out) and not out.startswith('Z')
if PARAR:
    try:
        pgid = int(open('/content/eglantine.pid').read())
        for sig in (signal.SIGTERM, signal.SIGKILL):
            try: os.killpg(pgid, sig)
            except ProcessLookupError: break
            time.sleep(3)
    except FileNotFoundError:
        pass
    me = {os.getpid(), os.getppid()}
    for line in subprocess.run(['ps','-eo','pid,args'],capture_output=True,text=True).stdout.splitlines()[1:]:
        pid, _, args = line.strip().partition(' ')
        if int(pid) not in me and any(k in args for k in ('bin/meson','ninja','build-eglantine.sh','build-eden-nx2-ps5.sh')):
            print('matando', pid, args[:100]); os.kill(int(pid), signal.SIGKILL)
    open('/content/eglantine.exit','w').write('137')
    print('Build parado. Para seguir, ejecuta la celda 2.')
else:
    pid = int(open('/content/eglantine.pid').read()) if os.path.exists('/content/eglantine.pid') else 0
    done = os.path.exists('/content/eglantine.exit')
    if done: print('Terminado, exit code:', open('/content/eglantine.exit').read().strip())
    elif pid and alive(pid): print('⏳ Compilando…')
    else: print('Sin build en marcha')
    if os.path.exists('/content/eglantine.log'):
        log = open('/content/eglantine.log',errors='replace').read()
        steps = re.findall(r'^==> .*$',log,re.M); n = re.findall(r'\[(\d+)/(\d+)\]',log[-20000:])
        if steps: print('Último paso:',steps[-1])
        if n: print('ninja: %s/%s' % n[-1])
        print('\n'.join(log.splitlines()[-15:]))

In [ ]:
# 4 · Guardar: ZIP de la app (Drive + descarga) y estado para la próxima vez
import os, shutil, datetime
DRV = '/content/drive/MyDrive/eglantine-ps5'
APP = 'PPSA99308'
app = f'/content/work-eglantine/ProsperoEden/build/release/{APP}'
assert os.path.exists('/content/eglantine.exit') and open('/content/eglantine.exit').read().strip()=='0', 'El build no ha terminado bien en este entorno: mira la celda 3.'
assert os.path.isdir(app), 'No hay carpeta de resultado; mira el log'
f = datetime.datetime.now().strftime('%Y%m%d-%H%M')
z = shutil.make_archive(f'/content/Eglantine-{APP}-{f}','zip',root_dir=os.path.dirname(app),base_dir=APP)
shutil.copy(z, DRV); shutil.copy('/content/eglantine.log', f'{DRV}/build-{f}.log')
from google.colab import files; files.download(z)
!rm -f /content/estado-eglantine.zip
!cd / && zip -q -r -y -1 /content/estado-eglantine.zip root/.cache content/work-eglantine usr/local/lib/libLLVMSPIRVLib* usr/local/lib/pkgconfig/LLVMSPIRVLib.pc usr/local/include/LLVMSPIRVLib usr/local/bin/llvm-spirv
!cp /content/estado-eglantine.zip "{DRV}/estado-eglantine.zip.tmp" && mv "{DRV}/estado-eglantine.zip.tmp" "{DRV}/estado-eglantine.zip"
print('✅ ZIP de la app y estado guardados en Drive (eglantine-ps5/)')